# v5: straight unmarked intersections and stronger sharp turns

This is an experimental camera-only controller. Run one cell at a time; driving is commented out. Shut down the previous notebook kernel before opening this camera. Copy your working HSV/black thresholds, speeds, gains, gap timeout, and green-turn timing before testing.

**Behaviour:** detected unmarked junctions continue straight even when side branches are visible. An ordinary bend with no straight continuation may still turn without a green marker. A detected T with no straight exit continues only for INTERSECTION_TIMEOUT, then stops instead of choosing a branch. Confirmed green instructions override straight junction handling. Sharp turns reverse a wheel and retain their direction while the line is temporarily lost, then require several aligned observations before exit.

**Limits:** branch geometry is based on image runs, not a calibrated map. Shadows, diagonal approaches, thick tape, or tight curves resembling a T can be misclassified. Verify junction/corner labels with the motors stopped. A missing green marker is indistinguishable from a missed colour detection. No obstacle/finish handling. Timed straight movement can run off the course if no exit exists; keep timeout short and clear space ahead. Hardware power alarms must be resolved. Software stops cannot protect against a frozen process/driver.

## 1. Initialize once

**Updated corner sequence:** confirm direction → forward advance → opposite-wheel pivot → leave the incoming alignment → reacquire the outgoing line for three frames. Calibrate advance so the wheel axle, not the camera, reaches the corner. Equal/opposite motor commands approximate an in-place pivot; wheel mismatch and slip can translate the robot. No encoder/IMU feedback measures 90°.


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Track geometry
Narrow candidates are associated near-to-far. Broad crossbars are excluded from line-centre averaging. Bilateral branches indicate a junction; a one-sided branch plus a plausible straight exit also indicates a junction. A one-sided branch without a straight exit indicates a possible sharp corner.


In [ ]:
# These fractions assume a forward/downward camera and an isolated black track.
BANDS = [('near', 0.72, 0.88), ('middle', 0.54, 0.70), ('far', 0.36, 0.52)]
MAX_LINE_WIDTH = 0.22
BRANCH_REACH = 0.16
STRAIGHT_CORRIDOR = 0.12
BRANCH_ROWS_REQUIRED = 2

def pixel_runs(values, minimum=3):
    runs = []
    start = None
    for x, value in enumerate(list(values) + [False]):
        if value and start is None:
            start = x
        elif not value and start is not None:
            if x - start >= minimum:
                runs.append((start, x - 1))
            start = None
    return runs

def band_candidates(mask, top, bottom):
    height, width = len(mask), len(mask[0])
    rows = mask[int(top * height):max(int(top * height) + 1, int(bottom * height))]
    votes = [sum(bool(row[x]) for row in rows) >= max(1, len(rows) * 0.25)
             for x in range(width)]
    return pixel_runs(votes)

def analyze_track(mask, previous_x=None):
    """Pure geometry; mask may be a Python list or an OpenCV mask converted to a list.

    Keep narrow line candidates instead of averaging a whole intersection bar.
    A bilateral bar is a junction; a one-sided bar needs a straight exit to be
    a junction. A one-sided bar with no straight exit is a possible corner.
    These are camera-image heuristics, not a map or proof of connectivity.
    """
    height, width = len(mask), len(mask[0])
    anchor = width / 2.0 if previous_x is None else previous_x
    positions, candidates = {}, {}
    for name, top, bottom in BANDS:
        runs = band_candidates(mask, top, bottom)
        narrow = [r for r in runs if r[1] - r[0] + 1 <= MAX_LINE_WIDTH * width]
        candidates[name] = narrow
        if narrow:
            chosen = min(narrow, key=lambda r: abs((r[0] + r[1]) / 2.0 - anchor))
            x = (chosen[0] + chosen[1]) / 2.0
            positions[name] = (x - width / 2.0) / (width / 2.0)
            # Associate farther candidates with the selected incoming path.
            anchor = x
        else:
            positions[name] = None
    entry = (width * (1.0 + positions['near']) / 2.0
             if positions['near'] is not None else
             (width / 2.0 if previous_x is None else previous_x))
    # Look above the branch-scanning region for a plausible straight continuation.
    upper = band_candidates(mask, 0.28, 0.43)
    straight_exit = any(r[1]-r[0]+1 <= MAX_LINE_WIDTH * width and
                        abs((r[0]+r[1])/2.0-entry) <= STRAIGHT_CORRIDOR*width for r in upper)
    both_rows = left_rows = right_rows = 0
    for y in range(int(height * 0.50), int(height * 0.76), 3):
        for lo, hi in pixel_runs(mask[y]):
            if not lo - 3 <= entry <= hi + 3:
                continue
            left = entry - lo >= BRANCH_REACH * width
            right = hi - entry >= BRANCH_REACH * width
            if left and right:
                both_rows += 1
            elif left:
                left_rows += 1
            elif right:
                right_rows += 1
    bilateral = both_rows >= BRANCH_ROWS_REQUIRED
    left = left_rows >= BRANCH_ROWS_REQUIRED
    right = right_rows >= BRANCH_ROWS_REQUIRED
    junction = bilateral or (straight_exit and (left or right))
    corner = 0
    if not junction and not straight_exit and left != right:
        corner = -1 if left else 1
    return dict(positions=positions, candidates=candidates, entry_x=entry,
                junction=junction, corner=corner, straight_exit=straight_exit)


## 3. Image and green-marker detection
The preview mask now displays BLACK tape. Green rectangles identify physical markers. Branch parameters are normalized to frame width except the green detector pixel limits.


In [ ]:
BLACK_THRESHOLD = 70

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65  # Require a nearby marker before initiating a turn.
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15

def detect_green(frame):
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ('far', 'middle', 'near') if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height):
    if not markers:
        return 'NONE'
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return 'UNKNOWN'
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return 'UNKNOWN'
    dx, dy = dx / length, dy / length
    sides = {'LEFT': [], 'RIGHT': []}
    for marker in markers:
        if marker['y'] < height * GREEN_TRIGGER_Y:
            continue
        score = (marker['x'] - near_x) * dy - (marker['y'] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if score > 0 else 'RIGHT'].append(marker)
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'

class TrackDetector:
    def __init__(self):
        self.previous_x = None

    def detect(self, frame):
        height, width = frame.shape[:2]
        markers, green_mask = detect_green(frame)
        gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5,5), 0)
        _, black_mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
        # Dark green tape must not become a false black branch.
        black_mask[green_mask > 0] = 0
        observation = analyze_track(black_mask.tolist(), self.previous_x)
        if observation['positions']['near'] is not None:
            self.previous_x = observation['entry_x']
        overlay = frame.copy()
        for name, top, bottom in BANDS:
            y0, y1 = int(height*top), int(height*bottom)
            cv2.rectangle(overlay,(0,y0),(width-1,y1),(255,0,0),1)
            for lo,hi in observation['candidates'][name]:
                cv2.rectangle(overlay,(lo,y0),(hi,y1),(255,255,0),1)
            value = observation['positions'][name]
            if value is not None:
                cv2.circle(overlay,(int(width*(1+value)/2),int((y0+y1)/2)),4,(0,255,0),-1)
        for marker in markers:
            x,y,w,h=marker['box']
            cv2.rectangle(overlay,(x,y),(x+w,y+h),(0,255,0),2)
        return observation, markers, overlay, black_mask, green_mask


## 4. Baseline and gap settings
Keep these for gap eligibility and bounded recovery. v5 overrides sharp-turn and intersection decisions below.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = False  # True enables an in-place turn with one wheel reversing.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. v5 intersection and sharp-turn controller
Copy tested reliable motor commands. Going slower helps approaches only while the motors still move reliably. Sharp turns pivot in place, with separate left/right speed and minimum-duration settings. Recent repeated sharp turns reduce subsequent forward speed; pivot torque is preserved. Never interpret minimum duration as an exact angle.


In [ ]:
# All speeds are normalized commands, not m/s. Copy reliable hardware settings.
INTERSECTION_SPEED = 0.13
INTERSECTION_TIMEOUT = 1.8
INTERSECTION_MIN_TIME = 0.25
INTERSECTION_CLEAR_FRAMES = 3
INTERSECTION_STEER_LIMIT = 0.025
SHARP_ENTER = 0.42
SHARP_CONFIRM_FRAMES = 2
# Advance until the wheel axle reaches the corner, then counter-rotate wheels.
# Initial guesses: calibrate distance on the floor; these are not metres/degrees.
SHARP_ADVANCE_SPEED = 0.13
SHARP_LEFT_ADVANCE_TIME = 0.20
SHARP_RIGHT_ADVANCE_TIME = 0.20
SHARP_LEFT_SPEED = 0.15
SHARP_RIGHT_SPEED = 0.16
SHARP_LEFT_MIN_TIME = 0.20
SHARP_RIGHT_MIN_TIME = 0.25
SHARP_TIMEOUT = 1.8
SHARP_REACQUIRE_FRAMES = 3
SHARP_HEADING_EXIT = 0.18
DENSE_CURVE_WINDOW = 6.0
DENSE_CURVE_COUNT = 2
DENSE_CURVE_SPEED = 0.13

def steering_error(positions):
    weights = {'near':0.45,'middle':0.35,'far':0.20}
    visible = [(k,v) for k,v in positions.items() if v is not None]
    if not visible:
        return None
    e = sum(weights[k]*v for k,v in visible)/sum(weights[k] for k,v in visible)
    look = positions.get('far')
    if look is None:look=positions.get('middle')
    if positions.get('near') is not None and look is not None:
        e += 0.40*(look-positions['near'])
    return clamp(e,-1.,1.)

class V5LineController(GapCurveController):
    def __init__(self):
        super().__init__()
        self.observation = {}
        self.junction_started = None
        self.junction_clear = 0
        self.junction_entry = 0.0
        self.sharp_candidate = 0
        self.sharp_frames = 0
        self.sharp_reacquired = 0
        self.sharp_events = []
        self.sharp_advance_started = None
        self.sharp_departed = False

    def pivot(self, direction):
        speed = SHARP_RIGHT_SPEED if direction > 0 else SHARP_LEFT_SPEED
        speed = clamp(speed,0.,MAX_SPEED)
        return (speed,-speed) if direction > 0 else (-speed,speed)

    def step(self, positions, now):
        if self.stopped:return 0.,0.,'STOP: controller stopped'
        junction = self.observation.get('junction',False)
        corner = self.observation.get('corner',0)
        self.sharp_events = [t for t in self.sharp_events if now-t <= DENSE_CURVE_WINDOW]

        # Direction is latched when confirmed. Advance even if tape disappears,
        # but only for this bounded interval. The run loop still checks camera age.
        if self.sharp_advance_started is not None:
            advance_time = (SHARP_RIGHT_ADVANCE_TIME if self.turn_direction > 0
                            else SHARP_LEFT_ADVANCE_TIME)
            if now - self.sharp_advance_started < advance_time:
                speed = clamp(SHARP_ADVANCE_SPEED, 0., MAX_SPEED)
                return speed, speed, 'SHARP_ADVANCE_RIGHT' if self.turn_direction > 0 else 'SHARP_ADVANCE_LEFT'
            self.sharp_advance_started = None
            self.turn_started = now  # Pivot timeout/minimum start AFTER advance.
            self.sharp_departed = False
            self.sharp_reacquired = 0

        # Once committed to a sharp turn, continue in that direction even if tape
        # leaves the image. Do not apply the old 0.35-second lost-line timeout.
        if self.turn_started is not None:
            elapsed = now-self.turn_started
            if elapsed >= SHARP_TIMEOUT:return self.stop('sharp turn timed out')
            near,middle = positions.get('near'),positions.get('middle')
            minimum = SHARP_RIGHT_MIN_TIME if self.turn_direction>0 else SHARP_LEFT_MIN_TIME
            centered = (near is not None and middle is not None and
                        abs(near)<TURN_EXIT and abs(middle)<TURN_EXIT and
                        abs(middle-near)<SHARP_HEADING_EXIT and not junction and not corner)
            if not centered:
                self.sharp_departed = True
            self.sharp_reacquired = (self.sharp_reacquired + 1
                                    if self.sharp_departed and centered and elapsed >= minimum else 0)
            if self.sharp_reacquired < SHARP_REACQUIRE_FRAMES:
                return (*self.pivot(self.turn_direction),'SHARP_RIGHT' if self.turn_direction>0 else 'SHARP_LEFT')
            self.turn_started=None
            self.turn_direction=0
            self.lost_started=None
            self.last_direction=0
            self.sharp_reacquired=0
            return 0.,0.,'SHARP_DONE'

        if self.junction_started is None and junction and self.gap_started is None:
            self.junction_started=now
            self.junction_entry=positions.get('near') or 0.0
            self.junction_clear=0
            self.straight_frames=0
            self.straight_seen=None
            self.sharp_frames=0

        if self.junction_started is not None:
            if now-self.junction_started >= INTERSECTION_TIMEOUT:
                return self.stop('no straight exit after intersection')
            near,middle=positions.get('near'),positions.get('middle')
            aligned=(near is not None and middle is not None and
                     abs(near-self.junction_entry)<0.25 and abs(middle-near)<0.18)
            clear=(not junction and not corner and aligned and
                   now-self.junction_started>=INTERSECTION_MIN_TIME)
            self.junction_clear=self.junction_clear+1 if clear else 0
            if self.junction_clear < INTERSECTION_CLEAR_FRAMES:
                # Ignore side-branch centroids. Correct only near incoming track,
                # and only within a tight corridor; otherwise hold equal commands.
                error=near if near is not None and abs(near-self.junction_entry)<0.25 else 0.0
                correction=clamp(KP*error,-INTERSECTION_STEER_LIMIT,INTERSECTION_STEER_LIMIT)
                return (clamp(INTERSECTION_SPEED+correction,0.,MAX_SPEED),
                        clamp(INTERSECTION_SPEED-correction,0.,MAX_SPEED),'INTERSECTION_STRAIGHT')
            self.junction_started=None
            self.last_direction=0
            self.lost_started=None

        error=steering_error(positions)
        request=corner or ((1 if error>0 else -1) if error is not None and abs(error)>=SHARP_ENTER else 0)
        if self.gap_started is None and request:
            self.sharp_frames=self.sharp_frames+1 if request==self.sharp_candidate else 1
            self.sharp_candidate=request
            if self.sharp_frames>=SHARP_CONFIRM_FRAMES:
                self.turn_started=now  # Also reserves this manoeuvre against green overrides.
                self.sharp_advance_started=now
                self.sharp_departed=False
                self.turn_direction=request
                self.last_direction=request
                self.sharp_reacquired=0
                self.sharp_frames=0
                self.straight_frames=0
                self.straight_seen=None
                self.sharp_events.append(now)
                speed=clamp(SHARP_ADVANCE_SPEED,0.,MAX_SPEED)
                return speed,speed,'SHARP_ADVANCE_RIGHT' if request>0 else 'SHARP_ADVANCE_LEFT'
            speed=min(BASE_SPEED,CURVE_SPEED)
            return speed,speed,'SHARP_APPROACH'
        else:
            self.sharp_frames=0
            self.sharp_candidate=0

        left,right,state=super().step(positions,now)
        # The inherited gap controller can begin a directional search when tape
        # disappears without straight-gap evidence. Bound it with SHARP_TIMEOUT.
        if self.turn_started is not None:
            # A loss-recovery search has no identified corner to advance onto.
            self.sharp_departed=True
            self.sharp_events.append(now)
            return (*self.pivot(self.turn_direction),'SHARP_SEARCH')
        if state=='FOLLOW':
            bend=0.0 if error is None else min(abs(error)/SHARP_ENTER,1.)
            base=BASE_SPEED+(min(BASE_SPEED,CURVE_SPEED)-BASE_SPEED)*bend
            if len(self.sharp_events)>=DENSE_CURVE_COUNT:
                base=min(base,DENSE_CURVE_SPEED)
                state='FOLLOW_SLOW'
            correction=KP*(error or 0.)
            left,right=clamp(base+correction,0.,MAX_SPEED),clamp(base-correction,0.,MAX_SPEED)
        return left,right,state


## 6. Green-marker priority
The repaired stop/advance/turn/cooldown state machine remains. Green confirmation can override INTERSECTION_STRAIGHT. GREEN_TURN_SPEED controls marker-requested turns; SHARP_RIGHT_SPEED controls ordinary sharp right bends.


In [ ]:
# Initial trial values, not calibrated turn angles or distances.
GREEN_CONFIRM_FRAMES = 3
GREEN_STOP_TIME = 0.12
GREEN_ADVANCE_TIME = 0.15 # Advance toward junction after stopping; tune from preview.
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_TURN_SPEED = 0.15
GREEN_TURN_MIN_TIME = 0.25
GREEN_TURN_TIMEOUT = 1.5
UTURN_MIN_TIME = 0.8
UTURN_TIMEOUT = 2.5
GREEN_REACQUIRE_FRAMES = 3
GREEN_CLEAR_FRAMES = 5
GREEN_COOLDOWN = 0.8

def line_centered(positions):
    return all(positions.get(name) is not None and abs(positions[name]) < TURN_EXIT
               for name in ('near', 'middle'))

class GreenNavigator:
    def __init__(self):
        self.line = V5LineController()
        self.state = 'FOLLOW'
        self.state_started = None
        self.direction = None
        self.candidate = None
        self.candidate_frames = 0
        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ''

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, 'STOP: ' + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self):
        speed = clamp(GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        if self.direction == 'RIGHT':
            return speed, -speed, 'GREEN_TURN_RIGHT'
        return -speed, speed, 'GREEN_U_TURN' if self.direction == 'BOTH' else 'GREEN_TURN_LEFT'

    def step(self, positions, direction, now):
        if self.stopped:
            return self.stop(self.reason)

        if self.state == 'GREEN_STOP':
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, 'GREEN_STOP'
            self.enter('GREEN_ADVANCE', now)

        if self.state == 'GREEN_ADVANCE':
            if now - self.state_started < GREEN_ADVANCE_TIME:
                speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
                return speed, speed, 'GREEN_ADVANCE'
            self.departed = False
            self.reacquire_frames = 0
            self.enter('GREEN_TURN', now)

        if self.state == 'GREEN_TURN':
            elapsed = now - self.state_started
            limit = UTURN_TIMEOUT if self.direction == 'BOTH' else GREEN_TURN_TIMEOUT
            minimum = UTURN_MIN_TIME if self.direction == 'BOTH' else GREEN_TURN_MIN_TIME
            if elapsed >= limit:
                return self.stop('green turn timed out')
            centered = line_centered(positions)
            if not centered:
                self.departed = True
            if self.departed and elapsed >= minimum and centered:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn()
            # Remove stale line-turn and gap history before handing control back.
            self.line = V5LineController()
            self.enter('FOLLOW', now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            return 0.0, 0.0, 'GREEN_DONE'

        if not self.armed:
            # Require actual absence, not UNKNOWN/CENTER, to clear the old marker.
            self.clear_frames = self.clear_frames + 1 if direction == 'NONE' else 0
            if now >= self.cooldown_until and self.clear_frames >= GREEN_CLEAR_FRAMES:
                self.armed = True

        # Do not initiate another manoeuvre in the middle of a gap or sharp turn.
        eligible = (self.armed and self.line.gap_started is None and
                    self.line.turn_started is None and direction in ('LEFT','RIGHT','BOTH'))
        if eligible:
            if direction == self.candidate:
                self.candidate_frames += 1
            else:
                self.candidate, self.candidate_frames = direction, 1
            if self.candidate_frames >= GREEN_CONFIRM_FRAMES:
                self.direction = direction
                self.armed = False
                self.clear_frames = 0
                self.candidate = None
                self.candidate_frames = 0
                self.line = V5LineController()
                self.enter('GREEN_STOP', now)
                return 0.0, 0.0, 'GREEN_STOP'
        else:
            self.candidate = None
            self.candidate_frames = 0

        left, right, state = self.line.step(positions, now)
        if self.line.stopped:
            return self.stop(state)
        return left, right, state


## 7. Run loop
Each run resets tracking/controller memory. Freshness is checked before duplicate-frame handling and again after processing.


In [ ]:
overlay_view=widgets.Image(format='jpeg',width=336,height=336)
mask_view=widgets.Image(format='jpeg',width=336,height=336)
status_view=widgets.Label(value='v5 ready; preview first')
display(widgets.VBox([widgets.HBox([overlay_view,mask_view]),status_view]))

def run_v5(seconds=10,drive=False):
    robot.stop()
    controller=GreenNavigator()
    detector=TrackDetector()
    deadline=time.monotonic()+seconds
    previous_stamp=None
    next_display=0.0
    try:
        while time.monotonic()<deadline:
            sample=latest['sample']
            if sample is None:
                print('Stopped: no camera frame');break
            frame,captured_at=sample
            now=time.monotonic()
            if now-captured_at>FRAME_TIMEOUT:
                print('Stopped: stale camera frame');break
            if captured_at==previous_stamp:
                time.sleep(0.005);continue
            previous_stamp=captured_at
            observation,markers,overlay,black_mask,green_mask=detector.detect(frame)
            positions=observation['positions']
            height,width=frame.shape[:2]
            direction=classify_green_direction(positions,markers,width,height)
            now=time.monotonic()
            if now>=deadline or now-captured_at>FRAME_TIMEOUT:
                print('Stopped: deadline or slow processing');break
            controller.line.observation=observation
            left,right,state=controller.step(positions,direction,now)
            if drive:robot.set_motors(left,right)
            if now>=next_display:
                label='%s | green=%s | junction=%s corner=%s | L=%.2f R=%.2f' % (
                    state,direction,observation['junction'],observation['corner'],left,right)
                status_view.value=('DRIVE | ' if drive else 'PREVIEW | ')+label
                cv2.putText(overlay,state,(4,18),cv2.FONT_HERSHEY_SIMPLEX,.42,(0,0,255),1)
                overlay_view.value=bgr8_to_jpeg(overlay)
                mask_view.value=bgr8_to_jpeg(cv2.cvtColor(black_mask,cv2.COLOR_GRAY2BGR))
                next_display=now+.1
            if controller.stopped:
                print(state);break
            time.sleep(.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):run_v5(seconds,drive=False)
def run_robot(seconds=2):run_v5(seconds,drive=True)


## 8. Stationary preview
Test a straight, left/right curve, cross, T, and straight-plus-one-side junction. At unmarked branches expect junction=True and INTERSECTION_STRAIGHT. At ordinary single-path sharp bends expect corner=-1/+1 or SHARP_APPROACH/SHARP_LEFT/SHARP_RIGHT. Move through the feature by hand; preview timeouts deliberately stop too. If classification is wrong, fix the mask/geometry before driving.


In [ ]:
run_preview(seconds=20)


## 9. Short driving trial
Begin with two seconds on a known straight-plus-side branch with a straight exit, then left and right corners, green turns, and gaps. Increase overall test duration gradually. Confirm missing straight exits cause a timeout stop on a clear test floor.


In [ ]:
# Uncomment deliberately; this moves the robot:
# run_robot(seconds=2)


## Tuning order

1. Copy your previously tested detector and speed settings. Keep CURVE_SPEED <= BASE_SPEED and above the reliable movement threshold.
2. Verify junction/corner classifications. BRANCH_REACH controls side-branch reach; MAX_LINE_WIDTH separates narrow tape from bars. Camera angle and floor visibility matter.
3. Right turn understeers: verify reverse wheel direction, then increase SHARP_RIGHT_SPEED slightly (e.g. 0.16 to 0.17). SHARP_RIGHT_MIN_TIME prevents very early exit; increase only after checking the overlay. A timeout means stop and inspect, not extend endlessly.
4. SHARP_ENTER controls anticipatory pivot entry (smaller enters earlier); SHARP_REACQUIRE_FRAMES prevents single-frame premature exit. Sharp-turn loss uses SHARP_TIMEOUT, not the old LOST_TIMEOUT.
5. DENSE_CURVE_SPEED caps forward speed after two sharp turns in six seconds. This is recent-turn memory, not a guarantee about upcoming track geometry. Do not set below reliable motor power.
6. INTERSECTION_TIMEOUT bounds straight crossing when the exit is not recovered. INTERSECTION_SPEED must move both wheels reliably. No arbitrary unmarked left/right choice is made for a detected junction.
7. Retest green LEFT/RIGHT/BOTH, cooldown, gaps, and a deliberately missing tape segment. Save successful parameters and failure frames.

## 10. Cleanup

### Advance-before-turn calibration

- Start with SHARP_LEFT_ADVANCE_TIME and SHARP_RIGHT_ADVANCE_TIME at 0.20 s. These apply to confirmed sharp turns. Increase by 0.05 s if the axle stops short; decrease if it overshoots.
- Use SHARP_ADVANCE_SPEED above the reliable wheel-motion threshold. Do not change speed and time together while calibrating.
- Green-requested turns already use STOP → ADVANCE → PIVOT. Tune GREEN_ADVANCE_TIME separately because green markers are detected at a different location.
- Pivot speed sets wheel power, not angle. SHARP_*_MIN_TIME prevents early completion, and SHARP_TIMEOUT is a failure limit, not a 90-degree timer.
- A lost-line recovery search does not advance blindly; only a confirmed sharp-turn request uses the advance phase. Unmarked intersections still continue straight.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
